# VoxSure: intent model training (Colab)
Trains the same fine-tuned DistilBERT used in the deployed app. Runtime > Change runtime type > **T4 GPU** is optional (CPU also works, just slower).

Steps: 1) install, 2) build dataset, 3) train, 4) download `intent_model.zip` and put its contents in `ml/artifacts/intent_model/` of the repo.

In [ ]:
!pip -q install transformers==4.47.1 scikit-learn jiwer
import os
os.makedirs('ml/scripts', exist_ok=True); os.makedirs('ml/data', exist_ok=True); os.makedirs('ml/artifacts', exist_ok=True)

In [ ]:
%%writefile ml/scripts/build_dataset.py
"""Build the VoxSure intent dataset (deterministic, seed=42).

Provenance: every seed utterance below was written by the project author for this
project (no scraped or third-party data, no licence issues). Seeds are expanded with
conversational fillers and plausible ASR-style corruptions. All variants of one seed
share a `group` id and are assigned to ONE split together, which prevents
near-duplicate leakage between train/val/test.

Outputs (ml/data/): train.jsonl, val.jsonl, test.jsonl, ambiguous_eval.jsonl, stats.json
"""
from __future__ import annotations

import json
import random
from collections import Counter
from pathlib import Path

SEED = 42
OUT = Path(__file__).resolve().parents[1] / "data"

SEEDS: dict[str, list[str]] = {
    "library_hours": [
        "What time does the library open?",
        "Until what time is the library open tonight?",
        "Is the library open on Sundays?",
        "What are the library timings during exams?",
        "Does the library stay open late on weekdays?",
        "When does the library close today?",
        "Library opening hours please",
        "Can I study in the library on public holidays?",
        "Tell me the reading room timings",
        "How late can I stay at the central library?",
        "Is the library open this weekend?",
        "What are the working hours of the library?",
        "Is the reading hall open twenty four hours during finals?",
        "At what time does the library shut on Saturdays?",
        "Are there special library hours in the vacation?",
        "I want to know when the library is open",
    ],
    "library_borrow": [
        "How many books can I borrow from the library?",
        "How do I renew a library book?",
        "What is the fine for returning a book late?",
        "I lost a library book, what should I do?",
        "How long can I keep a borrowed book?",
        "Can I issue a book with my student ID card?",
        "How do I reserve a book that is already checked out?",
        "Where do I return borrowed books?",
        "Can I access e-journals and online databases from home?",
        "How do I get a library membership card?",
        "What is the loan period for reference books?",
        "Can I extend the due date of my issued book?",
        "How do I pay my overdue library fine?",
        "I want to renew my books online",
        "Is there a limit on the number of books I can take out?",
        "How do I request a book that the library does not have?",
    ],
    "academic_calendar": [
        "When does the next semester start?",
        "What are the dates of the winter break?",
        "When is the last day of classes this term?",
        "Where can I find the academic calendar?",
        "When do the semester registrations begin?",
        "Is there a holiday on the national day?",
        "What is the last date to add or drop a course?",
        "When does the spring term begin?",
        "Tell me the important dates for this semester",
        "When is the convocation ceremony?",
        "When will the new academic year start?",
        "What is the schedule for the mid semester break?",
        "Can you tell me the last working day before vacation?",
        "When is the course registration deadline?",
        "How many holidays do we get in the semester?",
        "When do classes resume after the break?",
    ],
    "exam_info": [
        "When are the final exams?",
        "How do I download my hall ticket?",
        "Where can I see the exam timetable?",
        "When will the exam results be declared?",
        "How do I apply for revaluation of my paper?",
        "What is the procedure to apply for a supplementary exam?",
        "What should I carry to the examination hall?",
        "How can I check my semester results?",
        "What happens if I miss an exam because of illness?",
        "Is a calculator allowed in the exam?",
        "When is the date sheet for end semester exams released?",
        "How do I get my admit card for the test?",
        "Can I see my marks for the internal assessment?",
        "How do I apply for a re-exam?",
        "What is the passing criteria for a course?",
        "Where do I check my seat number for the exam?",
    ],
    "fee_payment": [
        "How do I pay my tuition fees?",
        "What is the last date to pay the semester fee?",
        "Can I pay my fees in instalments?",
        "Is there a late fee for delayed payment?",
        "How do I get a fee receipt?",
        "My fee payment failed but money was deducted, what now?",
        "Can I pay the fees online?",
        "Where can I find the fee structure?",
        "How do I get a refund of my fees?",
        "Which payment methods are accepted for fees?",
        "I want to know the hostel and tuition fee amount",
        "How do I check whether my fees are paid?",
        "Is there a fee waiver or concession for siblings?",
        "What is the due date for the next fee instalment?",
        "I need a duplicate fee receipt",
        "Where do I go to pay fees in person?",
    ],
    "hostel_info": [
        "How do I apply for a hostel room?",
        "What are the hostel curfew timings?",
        "Can I change my hostel room?",
        "What facilities are available in the hostel?",
        "Are visitors allowed inside the hostel?",
        "How much is the hostel rent per semester?",
        "How do I report a maintenance problem in my hostel room?",
        "Is the hostel mess food included in the fee?",
        "What is the hostel check-out procedure before vacation?",
        "Do the hostels have laundry service?",
        "Can first year students stay in the hostel?",
        "What are the hostel rules about guests?",
        "How do I apply for a single room in the hostel?",
        "Who is the warden I should contact?",
        "When will the hostel allotment list be out?",
        "Is there a late entry permission for the hostel?",
    ],
    "it_helpdesk": [
        "I forgot my student portal password",
        "How do I connect to the campus wifi?",
        "The wifi is not working in my hostel",
        "How do I reset my university email password?",
        "I can't log in to the student portal",
        "Who do I contact for IT support?",
        "How do I get access to the computer lab?",
        "My campus email account is locked",
        "How can I install the licensed software provided by the university?",
        "The internet is very slow in the classroom block",
        "How do I set up the campus VPN?",
        "Where is the IT helpdesk located?",
        "How do I register my laptop for the campus network?",
        "The learning management system is not loading for me",
        "How do I raise a ticket with the helpdesk?",
        "I am unable to access my online course materials",
    ],
    "placement_process": [
        "How does the campus placement process work?",
        "When does the placement season start?",
        "Which companies visit the campus for recruitment?",
        "How do I register for placements?",
        "What are the eligibility criteria for campus placements?",
        "Can I sit for placements if I have a backlog?",
        "How do I upload my resume to the placement portal?",
        "Are there mock interviews or training before placements?",
        "How do I apply for an internship through the placement cell?",
        "Who do I contact in the training and placement office?",
        "What is the average package offered last year?",
        "Can I attend more than one company's selection process?",
        "How are students shortlisted for interviews?",
        "I want to know about the pre placement talks",
        "What documents do I need for the placement drive?",
        "Is there a minimum CGPA to be eligible for placement?",
    ],
    "scholarship_aid": [
        "What scholarships are available for students?",
        "How do I apply for a merit scholarship?",
        "What is the deadline for scholarship applications?",
        "Am I eligible for financial aid?",
        "What documents are needed for a scholarship application?",
        "How do I check my scholarship status?",
        "Is there a scholarship for students from low income families?",
        "How can I get an education loan through the university?",
        "Can I get a fee waiver based on my marks?",
        "Where do I submit the scholarship form?",
        "When will the scholarship amount be credited?",
        "Are there any sports quota scholarships?",
        "Does the university offer need based financial assistance?",
        "How do I renew my scholarship for the next year?",
        "Who handles scholarship queries?",
        "I would like to know about government scholarships",
    ],
    "certificates": [
        "How do I get my transcript?",
        "How can I apply for a bonafide certificate?",
        "How long does it take to get a degree certificate?",
        "I need a character certificate, how do I apply?",
        "How do I request a duplicate marksheet?",
        "Can I get my transcripts sent to another university?",
        "What is the fee for an official transcript?",
        "Where do I collect my provisional certificate?",
        "I need a letter confirming that I am a student here",
        "How do I get a no objection certificate for an internship?",
        "How do I apply for the migration certificate?",
        "Can I get my documents attested on campus?",
        "How do I apply for a transfer certificate?",
        "What is the process to get a medium of instruction letter?",
        "I lost my original degree, can I get a duplicate?",
        "How do I get my id card replaced?",
    ],
    "campus_services": [
        "What time does the canteen open?",
        "Where is the campus medical center?",
        "Is there a bus service from the campus to the city?",
        "Where can I find the sports complex timings?",
        "How do I book the auditorium for an event?",
        "Is there an ATM on the campus?",
        "Where is the nearest pharmacy on campus?",
        "What are the gym timings?",
        "Is there a shuttle between the hostel and the academic block?",
        "How do I join a student club?",
        "Where is the campus health clinic and when is it open?",
        "Is there a bookstore or stationery shop on campus?",
        "Where can I park my bicycle or bike?",
        "How do I get a campus bus pass?",
        "What food options are available in the food court?",
        "Does the campus have a counselling service?",
    ],
    "greeting": [
        "Hello",
        "Hi there",
        "Hey VoxSure",
        "Good morning",
        "Good afternoon",
        "Good evening",
        "Hello, are you there?",
        "Hi, I need some help",
        "Hey, can you help me?",
        "Namaste",
        "Hi assistant",
        "Hello there, how are you?",
    ],
    "thanks_bye": [
        "Thank you",
        "Thanks a lot",
        "That was helpful, thanks",
        "Goodbye",
        "Bye bye",
        "See you later",
        "Thanks, that's all I needed",
        "Okay, thank you so much",
        "Great, thanks for the help",
        "Alright, bye",
        "Cheers, that is all",
        "Thank you very much, have a good day",
    ],
    "out_of_scope": [
        "What is the weather like today?",
        "Tell me a joke",
        "Who won the cricket match yesterday?",
        "What is the price of bitcoin?",
        "How do I cook pasta?",
        "Can you write my essay for me?",
        "What is the capital of France?",
        "Play some music",
        "Book me a flight to Mumbai",
        "What are the latest news headlines?",
        "Who is the president of the United States?",
        "How do I lose weight fast?",
        "Order me a pizza",
        "Translate this sentence into Spanish",
        "What is the meaning of life?",
        "Can you give me a stock tip?",
        "Solve this calculus problem for me",
        "How do I fix my car engine?",
        "What movies are playing this weekend?",
        "Tell me about the history of Rome",
        "What is the best phone to buy?",
        "Give me a recipe for chocolate cake",
        "What is my horoscope today?",
        "How do I get a visa for Canada?",
        "Can you diagnose my headache?",
        "Who is the best football player in the world?",
        "Set an alarm for six in the morning",
        "What is the exchange rate for the dollar?",
        "Write a python program to sort a list",
        "Where can I buy a cheap laptop?",
        "How tall is Mount Everest?",
        "Recommend a good restaurant in town",
        "Explain how black holes work",
        "What is the speed of light?",
        "Can you call my mother?",
        "Sing me a song",
        "Do you know any good vacation spots?",
        "How do I file my income tax return?",
        "Tell me the lyrics of a popular song",
        "What should I name my new puppy?",
    ],
}

# Plausible ASR confusions applied to some variants (word -> replacement).
ASR_SUBS = {
    "library": "libary", "hostel": "hostile", "fees": "feeds", "fee": "free",
    "exam": "example", "exams": "examples", "wifi": "why fi", "semester": "seminar",
    "scholarship": "scholar ship", "scholarships": "scholar ships", "transcript": "transcribe",
    "placement": "placements", "placements": "placement", "renew": "renew you",
    "bonafide": "bona fide", "canteen": "can teen", "hall": "hole", "ticket": "tick it",
    "borrow": "barrow", "books": "box", "deadline": "dead line", "timings": "timing",
    "result": "results", "results": "result", "campus": "camp us", "calendar": "calender",
    "password": "pass word", "portal": "pour tal", "resume": "resume a", "warden": "warding",
    "curfew": "curfew", "convocation": "convocation", "revaluation": "re evaluation",
}
PREFIXES = ["", "", "", "um ", "hey ", "can you tell me ", "i want to know ", "please tell me ",
            "quick question ", "excuse me ", "so ", "hi, ", "okay so "]
SUFFIXES = ["", "", "", " please", " thanks", " right now", " if you know", " today"]


def normalise_asr(text: str) -> str:
    """ASR-style: lower case, no punctuation."""
    return "".join(c for c in text.lower() if c.isalnum() or c.isspace() or c == "'").strip()


def corrupt(text: str, rng: random.Random) -> str:
    words = normalise_asr(text).split()
    out = []
    changed = False
    for w in words:
        if w in ASR_SUBS and rng.random() < 0.6:
            out.append(ASR_SUBS[w]); changed = True
        elif len(words) > 5 and rng.random() < 0.06:
            changed = True  # dropped word
        else:
            out.append(w)
    if not changed and len(words) > 4:  # guarantee some noise: drop one interior word
        out.pop(rng.randrange(1, len(out) - 1))
    return " ".join(out)


def variants(seed: str, rng: random.Random, n: int) -> list[str]:
    base = seed
    res = [base]
    seen = {base.lower()}
    tries = 0
    while len(res) < n and tries < 60:
        tries += 1
        mode = rng.random()
        core = base[0].lower() + base[1:] if len(base) > 1 else base
        if mode < 0.35:
            v = normalise_asr(base)
        elif mode < 0.65:
            v = normalise_asr(rng.choice(PREFIXES) + core.rstrip("?.") + rng.choice(SUFFIXES))
        else:
            v = corrupt(rng.choice(PREFIXES) + core.rstrip("?."), rng)
        v = " ".join(v.split())
        if v and v.lower() not in seen:
            seen.add(v.lower()); res.append(v)
    return res


# Hand-written ambiguous utterances: the *correct behaviour* is to clarify rather than
# answer. `plausible` lists the intents a reasonable reader could mean. Used ONLY for
# evaluation (never for training).
AMBIGUOUS = [
    ("When does it open?", ["library_hours", "campus_services"]),
    ("What time does it close?", ["library_hours", "campus_services"]),
    ("I need help with my form", ["scholarship_aid", "certificates", "fee_payment"]),
    ("Tell me about the dates", ["academic_calendar", "exam_info"]),
    ("When is the deadline?", ["fee_payment", "scholarship_aid", "academic_calendar"]),
    ("How do I apply?", ["scholarship_aid", "hostel_info", "placement_process"]),
    ("What is the fine?", ["library_borrow", "fee_payment"]),
    ("I want to renew", ["library_borrow", "scholarship_aid"]),
    ("Where do I go for that?", ["certificates", "it_helpdesk", "campus_services"]),
    ("Is it free?", ["campus_services", "fee_payment"]),
    ("I have a problem with my account", ["it_helpdesk", "fee_payment"]),
    ("Can I get a receipt?", ["fee_payment", "certificates"]),
    ("How long does it take?", ["certificates", "exam_info"]),
    ("Who do I contact?", ["it_helpdesk", "hostel_info", "placement_process"]),
    ("What are the timings?", ["library_hours", "campus_services", "hostel_info"]),
    ("I lost my card", ["library_borrow", "certificates"]),
    ("Tell me about the results", ["exam_info", "placement_process"]),
    ("Is it allowed?", ["exam_info", "hostel_info"]),
    ("When will it be credited?", ["scholarship_aid", "fee_payment"]),
    ("I need a letter", ["certificates", "placement_process"]),
    ("What documents do I need?", ["placement_process", "scholarship_aid", "certificates"]),
    ("Check my status", ["scholarship_aid", "fee_payment", "exam_info"]),
    ("The portal is not showing it", ["it_helpdesk", "exam_info", "fee_payment"]),
    ("Can I change it?", ["hostel_info", "academic_calendar"]),
    ("What's the last date?", ["academic_calendar", "fee_payment", "scholarship_aid"]),
    ("I want information about registration", ["academic_calendar", "placement_process"]),
    ("Do I need permission?", ["hostel_info", "certificates", "placement_process"]),
    ("Is there a limit?", ["library_borrow", "placement_process"]),
    ("When is the next one?", ["academic_calendar", "exam_info", "placement_process"]),
    ("Tell me more about the process", ["placement_process", "certificates", "hostel_info"]),
    ("Is it open on weekends?", ["library_hours", "campus_services"]),
    ("How much does it cost?", ["fee_payment", "hostel_info", "certificates"]),
    ("I want to apply for it", ["scholarship_aid", "hostel_info", "placement_process"]),
    ("When do I get it back?", ["library_borrow", "exam_info", "certificates"]),
    ("It is not working", ["it_helpdesk", "campus_services"]),
    ("Where is the office?", ["certificates", "fee_payment", "placement_process"]),
    ("Can someone help me with my request?", ["it_helpdesk", "certificates", "fee_payment"]),
    ("Is there any update?", ["exam_info", "scholarship_aid", "hostel_info"]),
    ("Tell me the rules", ["hostel_info", "library_borrow", "exam_info"]),
    ("Do I have to pay for that?", ["fee_payment", "certificates", "campus_services"]),
    ("What is the procedure?", ["certificates", "exam_info", "placement_process"]),
    ("How early should I arrive?", ["exam_info", "placement_process"]),
    ("I missed the date", ["academic_calendar", "fee_payment", "scholarship_aid"]),
    ("Can I extend it?", ["library_borrow", "academic_calendar", "hostel_info"]),
    ("Who is in charge?", ["hostel_info", "placement_process", "it_helpdesk"]),
    ("I need to reset it", ["it_helpdesk", "hostel_info"]),
    ("Is the form available online?", ["scholarship_aid", "certificates", "exam_info"]),
    ("Where can I see the list?", ["hostel_info", "placement_process", "exam_info"]),
    ("When does it start?", ["academic_calendar", "placement_process", "campus_services"]),
    ("What happens if I am late?", ["library_borrow", "fee_payment", "exam_info"]),
]


def main() -> None:
    rng = random.Random(SEED)
    OUT.mkdir(parents=True, exist_ok=True)
    groups: list[dict] = []
    gid = 0
    for intent, seeds in SEEDS.items():
        for s in seeds:
            n = 7 if intent not in ("greeting", "thanks_bye") else 5
            if intent == "out_of_scope":
                n = 6
            for v in variants(s, rng, n):
                groups.append({"group": gid, "intent": intent, "text": v, "seed": s})
            gid += 1

    # split by group within each intent: 70/15/15
    by_intent: dict[str, list[int]] = {}
    for g in groups:
        by_intent.setdefault(g["intent"], [])
        if g["group"] not in by_intent[g["intent"]]:
            by_intent[g["intent"]].append(g["group"])
    split_of: dict[int, str] = {}
    for intent, gids in by_intent.items():
        rng.shuffle(gids)
        n = len(gids)
        n_test, n_val = max(2, round(n * 0.15)), max(2, round(n * 0.15))
        for i, g in enumerate(gids):
            split_of[g] = "test" if i < n_test else "val" if i < n_test + n_val else "train"

    splits: dict[str, list[dict]] = {"train": [], "val": [], "test": []}
    for g in groups:
        splits[split_of[g["group"]]].append(
            {"text": g["text"], "intent": g["intent"], "group": g["group"]})
    for name, rows in splits.items():
        rng.shuffle(rows)
        with (OUT / f"{name}.jsonl").open("w", encoding="utf-8") as f:
            for r in rows:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
    amb = list(AMBIGUOUS)
    rng.shuffle(amb)
    half = len(amb) // 2
    for name, rows in (("ambiguous_val", amb[:half]), ("ambiguous_test", amb[half:])):
        with (OUT / f"{name}.jsonl").open("w", encoding="utf-8") as f:
            for text, plausible in rows:
                f.write(json.dumps({"text": text, "plausible": plausible}) + "\n")

    # leakage check: no identical normalised text across splits
    texts = {k: {normalise_asr(r["text"]) for r in v} for k, v in splits.items()}
    overlap = {f"{a}&{b}": len(texts[a] & texts[b])
               for a, b in (("train", "val"), ("train", "test"), ("val", "test"))}
    stats = {"seed": SEED, "n_seed_utterances": gid, "sizes": {k: len(v) for k, v in splits.items()},
             "per_class": {k: dict(Counter(r["intent"] for r in v)) for k, v in splits.items()},
             "exact_overlap_between_splits": overlap, "n_ambiguous_eval": len(AMBIGUOUS)}
    (OUT / "stats.json").write_text(json.dumps(stats, indent=2), encoding="utf-8")
    print(json.dumps(stats["sizes"]), "overlap:", overlap)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile ml/scripts/train_intent.py
"""Fine-tune DistilBERT for VoxSure intent classification.

Usage: python ml/scripts/train_intent.py [--epochs 6] [--seed 42]
Saves model, tokenizer, label_map.json and train_config.json (incl. fitted
temperature) to ml/artifacts/intent_model/. Temperature scaling is fitted on the
VALIDATION split only; the test split is never touched here.
"""
from __future__ import annotations

import argparse
import json
import random
import time
from pathlib import Path

import numpy as np
import torch
from torch import nn
from transformers import AutoModelForSequenceClassification, AutoTokenizer

ROOT = Path(__file__).resolve().parents[1]
DATA = ROOT / "data"
OUT = ROOT / "artifacts" / "intent_model"
BASE = "distilbert-base-uncased"
MAX_LEN = 48


def read(name: str) -> list[dict]:
    return [json.loads(l) for l in (DATA / f"{name}.jsonl").read_text(encoding="utf-8").splitlines()]


def encode(tok, texts):
    return tok(texts, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt")


@torch.no_grad()
def logits_for(model, tok, texts, bs=64):
    model.eval()
    out = []
    for i in range(0, len(texts), bs):
        out.append(model(**encode(tok, texts[i:i + bs])).logits)
    return torch.cat(out)


def fit_temperature(logits: torch.Tensor, labels: torch.Tensor) -> float:
    log_t = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=100)
    ce = nn.CrossEntropyLoss()

    def closure():
        opt.zero_grad()
        loss = ce(logits / log_t.exp(), labels)
        loss.backward()
        return loss

    opt.step(closure)
    return float(log_t.exp().item())


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--epochs", type=int, default=6)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--lr", type=float, default=5e-5)
    ap.add_argument("--bs", type=int, default=16)
    args = ap.parse_args()

    random.seed(args.seed); np.random.seed(args.seed); torch.manual_seed(args.seed)
    train, val = read("train"), read("val")
    labels = sorted({r["intent"] for r in train})
    l2i = {l: i for i, l in enumerate(labels)}

    tok = AutoTokenizer.from_pretrained(BASE)
    model = AutoModelForSequenceClassification.from_pretrained(
        BASE, num_labels=len(labels),
        id2label={i: l for l, i in l2i.items()}, label2id=l2i)
    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=0.01)
    steps = args.epochs * ((len(train) + args.bs - 1) // args.bs)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min((s + 1) / (0.1 * steps), max(0.0, (steps - s) / (0.9 * steps))))

    vt = [r["text"] for r in val]
    vy = torch.tensor([l2i[r["intent"]] for r in val])
    history, best_acc, best_state = [], -1.0, None
    t0 = time.time()
    for ep in range(args.epochs):
        model.train()
        random.shuffle(train)
        total = 0.0
        for i in range(0, len(train), args.bs):
            b = train[i:i + args.bs]
            enc = encode(tok, [r["text"] for r in b])
            y = torch.tensor([l2i[r["intent"]] for r in b])
            loss = model(**enc, labels=y).loss
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); opt.zero_grad()
            total += loss.item() * len(b)
        acc = float((logits_for(model, tok, vt).argmax(1) == vy).float().mean())
        history.append({"epoch": ep + 1, "train_loss": total / len(train), "val_acc": acc})
        print(history[-1], f"{time.time() - t0:.0f}s", flush=True)
        if acc >= best_acc:
            best_acc = acc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)

    temperature = fit_temperature(logits_for(model, tok, vt), vy)
    OUT.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(OUT, safe_serialization=True)
    tok.save_pretrained(OUT)
    (OUT / "label_map.json").write_text(json.dumps(labels, indent=2), encoding="utf-8")
    (OUT / "train_config.json").write_text(json.dumps({
        "base_model": BASE, "seed": args.seed, "epochs": args.epochs, "lr": args.lr,
        "batch_size": args.bs, "max_len": MAX_LEN, "temperature": temperature,
        "n_train": len(train), "n_val": len(val), "best_val_acc": best_acc,
        "history": history, "model_version": "distilbert-intent-v1",
    }, indent=2), encoding="utf-8")
    print(f"saved to {OUT}; temperature={temperature:.3f}; best val acc={best_acc:.3f}")


if __name__ == "__main__":
    main()


In [ ]:
!python ml/scripts/build_dataset.py

In [ ]:
!python ml/scripts/train_intent.py --epochs 6 --seed 42

## Quick held-out check (test split, never used for training or tuning)

In [ ]:
import json, torch, numpy as np
from sklearn.metrics import f1_score, accuracy_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification
d='ml/artifacts/intent_model'
tok=AutoTokenizer.from_pretrained(d); m=AutoModelForSequenceClassification.from_pretrained(d).eval()
labels=json.load(open(d+'/label_map.json')); l2i={l:i for i,l in enumerate(labels)}
test=[json.loads(l) for l in open('ml/data/test.jsonl')]
with torch.no_grad():
    pred=[]
    for i in range(0,len(test),64):
        b=tok([r['text'] for r in test[i:i+64]],padding=True,truncation=True,max_length=48,return_tensors='pt')
        pred+=m(**b).logits.argmax(1).tolist()
y=[l2i[r['intent']] for r in test]
print('accuracy',accuracy_score(y,pred),'macro-F1',f1_score(y,pred,average='macro'))

In [ ]:
!cd ml/artifacts && zip -qr intent_model.zip intent_model
from google.colab import files
files.download('ml/artifacts/intent_model.zip')